<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 01 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Start Doris and Build a Baseline Dataset</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Start a reusable single-node integrated Doris sandbox, inspect an ecommerce event dataset in S3, and build a verified 10,158,080-row internal table.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 · FE · BE · S3 TVF · Internal Table · Analytical Query</span>
</div>

By the end of this lab, you will have a running Doris environment, an `events` table containing more than 10 million rows, and analytical results produced from that table. Run the cells in order.

### Initialize the Lab

Run the next initialization cell before starting Section 1. It loads the local Lab helper, installs the output styles, and creates the `lab` object used by every later code cell.

You must run it again whenever you restart the Jupyter kernel. It does **not** start Docker, access S3, or change any data. When a later SQL cell runs, the helper can reconnect automatically to an existing Doris sandbox on `127.0.0.1:9030` and restore the `doris_course` database context. On a first run, complete Section 2 to create and start the sandbox. When initialization succeeds, a green **Lab tools are ready** message appears. Do not continue until you see that message.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


## 1. Understand the sandbox and dataset

The notebook connects to Frontend (FE) through FE's MySQL-compatible query port. FE accepts and analyzes SQL, creates and optimizes a distributed query plan, and assigns its plan fragments to Backend (BE) nodes. BE nodes execute the assigned plan fragments and, in this integrated storage-compute architecture, store internal-table data.

| Component | Role in this lab |
|---|---|
| Notebook client | Sends SQL to FE port 9030. |
| Frontend (FE) | Accepts and analyzes SQL, creates and optimizes a distributed query plan, and assigns its plan fragments. |
| Backend (BE) | Executes the assigned plan fragments and stores the `events` internal-table data. |

The official All-in-One image runs one FE and one BE in one container. It is a **single-node integrated Doris sandbox** for learning, not a production deployment. Docker named volumes keep the database metadata and table data after the container stops.

The dataset contains 10,158,080 ecommerce behavior records. Each row describes a `view`, `cart`, or `purchase` event. The `region` values are synthetic and remain stable for each user; they do not represent real user locations. `revenue` records the product price for purchases and `0.00` for the other event types.

<div style="max-width:920px;border:1px solid #dbe4e8;border-left:3px solid #d97706;border-radius:4px;background:#fffbeb;color:#334155;padding:10px 12px;margin:14px 0"><strong style="color:#17212b;display:block;margin-bottom:2px">Before you begin</strong>Allocate at least 4 CPU cores and 8 GB of memory to Docker, and keep at least 20 GB of disk space free. The first Docker image pull and S3 import can take several minutes.</div>

### Dataset source and attribution

This Lab uses a transformed subset of **eCommerce behavior data from multi category store**, published by Michael Kechinov and provided by the **REES46 Marketing Platform**. Thanks to REES46 for making the original ecommerce behavior data available.

- Original dataset: [eCommerce behavior data from multi category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/data)
- Dataset provider: [REES46 public datasets](https://rees46.com/en/datasets)
- Public Parquet mirror used to prepare the course files: [Firebolt E-Commerce Analytics Primer](https://www.firebolt.io/free-sample-datasets/e-commerce)

The files read in this Lab are a transformed copy stored in the course S3 bucket, not the original publication. The transformation preserves `event_time`, `event_type`, `product_id`, and `user_id`; creates a stable `event_id`; assigns each user to one of eight synthetic `region` values; and stores the source price as `revenue` only for `purchase` events. Synthetic regions do not represent real user locations.


## 2. Prepare the environment

This section verifies Docker, creates reusable storage, starts Doris 4.1.3, and connects to FE. The commands support macOS Apple Silicon and Linux x86_64. Docker selects the native image architecture automatically, so do not add `--platform linux/amd64` on Apple Silicon.

<div style="max-width:100%;overflow-x:auto;margin:16px 0 20px;padding-bottom:4px">
<table style="border-collapse:separate;border-spacing:6px 0;min-width:1180px;width:100%;table-layout:fixed;margin:0">
<tr>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>1 · Verify Docker</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Check the client and running daemon</span></td>
<td style="width:24px;text-align:center;vertical-align:middle;border:0;color:#94a3b8;font-size:22px;padding:0">&#8594;</td>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>2 · Check ports</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Inspect ports 8030, 8040, and 9030</span></td>
<td style="width:24px;text-align:center;vertical-align:middle;border:0;color:#94a3b8;font-size:22px;padding:0">&#8594;</td>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>3 · Create storage</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Create the network and named volumes</span></td>
<td style="width:24px;text-align:center;vertical-align:middle;border:0;color:#94a3b8;font-size:22px;padding:0">&#8594;</td>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>4 · Pull the image</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Download the pinned Doris 4.1.3 image</span></td>
<td style="width:24px;text-align:center;vertical-align:middle;border:0;color:#94a3b8;font-size:22px;padding:0">&#8594;</td>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>5 · Start Doris</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Start the container and wait for health</span></td>
<td style="width:24px;text-align:center;vertical-align:middle;border:0;color:#94a3b8;font-size:22px;padding:0">&#8594;</td>
<td style="width:165px;vertical-align:top;border:1px solid #dbe4e8;border-top:3px solid #0f766e;border-radius:4px;padding:10px;background:#f8fbfb"><strong>6 · Inspect the nodes</strong><br><span style="color:#64748b;font-size:12px;line-height:1.45">Connect to FE and query FE/BE status</span></td>
</tr>
</table>
</div>

The variables at the top of the next cell control the image, container, network, volumes, and host ports. Keep the values consistent with later connection commands if you change them.

**Docker installation and storage**

| Host | Docker runtime used by this Lab | Doris image selected automatically |
|---|---|---|
| macOS Apple Silicon | Docker Desktop for Mac with Apple silicon | `linux/arm64` |
| Linux x86_64 | Docker Engine for the Linux distribution | `linux/amd64` |

Both hosts pull the same `apache/doris:all-in-one-4.1.3` tag. The tag contains both architectures, and Docker selects the native one. Other operating-system and architecture combinations stop before any Docker resource is created.

Install Docker before starting. Docker installation can require administrator approval, a Docker Desktop first-run agreement, or organization-specific configuration. On macOS, the environment cell can open an installed Docker Desktop application when its daemon is stopped.

Docker images are not downloaded into the notebook directory. Docker stores images in its own managed data area; named volumes store FE metadata and BE table data. The volumes are named `doris-fe-meta` and `doris-be-storage`. You normally should not change their physical location for this Lab. If Docker's managed disk does not have enough free space, change Docker Desktop's disk-image location on macOS or Docker Engine's data-root on Linux before starting the Lab.

In [ ]:
lab.shell(r"""
set -euo pipefail

DORIS_IMAGE="apache/doris:all-in-one-4.1.3"
DORIS_CONTAINER="doris"
DORIS_NETWORK="doris-course"
FE_VOLUME="doris-fe-meta"
BE_VOLUME="doris-be-storage"

echo "[1/6] Verify the host, Docker client, context, and daemon"
HOST_OS=$(uname -s)
HOST_ARCH=$(uname -m)
case "${HOST_OS}/${HOST_ARCH}" in
  Darwin/arm64|Darwin/aarch64)
    ;;
  Linux/x86_64|Linux/amd64)
    ;;
  *)
    echo "Unsupported host: ${HOST_OS}/${HOST_ARCH}"
    echo "This Lab supports macOS Apple Silicon and Linux x86_64."
    exit 2
    ;;
esac

if ! command -v docker >/dev/null 2>&1; then
  echo "Docker CLI was not found. Install Docker for ${HOST_OS}/${HOST_ARCH}, restart the terminal or kernel, and rerun this cell."
  exit 3
fi

echo "docker_cli=$(docker --version)"
echo "docker_context=$(docker context show 2>/dev/null || echo unknown)"
if [ -n "${DOCKER_HOST:-}" ]; then
  echo "DOCKER_HOST=${DOCKER_HOST}"
fi

if ! docker info >/dev/null 2>&1; then
  if [ "$HOST_OS" = "Darwin" ]; then
    if [ ! -d /Applications/Docker.app ]; then
      echo "Docker Desktop is not installed in /Applications/Docker.app."
      echo "Install the Apple silicon build of Docker Desktop, then rerun this cell."
      exit 4
    fi
    echo "Docker Desktop is installed but its daemon is not ready. Opening Docker Desktop..."
    open -a Docker
    for attempt in $(seq 1 60); do
      if docker info >/dev/null 2>&1; then
        break
      fi
      if [ $((attempt % 5)) -eq 0 ]; then
        echo "Waiting for Docker Desktop (${attempt}/60)..."
      fi
      sleep 2
    done
  else
    echo "Docker Engine is installed but its daemon is unavailable."
    echo "Start it in a terminal, commonly with: sudo systemctl start docker"
    echo "For rootless Docker, commonly use: systemctl --user start docker"
    exit 5
  fi
fi

if ! docker info >/dev/null 2>&1; then
  echo "Docker did not become ready. Complete any first-run dialog, check the selected context and DOCKER_HOST, then rerun this cell."
  exit 6
fi

docker version --format 'client={{.Client.Version}} server={{.Server.Version}}'
docker info --format 'cpus={{.NCPU}} memory={{.MemTotal}} docker_root={{.DockerRootDir}} os={{.OperatingSystem}} arch={{.Architecture}}'

echo "[2/6] Inspect ports 8030, 8040, and 9030"
if command -v lsof >/dev/null 2>&1; then
  lsof -nP -iTCP:8030 -iTCP:8040 -iTCP:9030 -sTCP:LISTEN || true
elif command -v ss >/dev/null 2>&1; then
  ss -ltn '( sport = :8030 or sport = :8040 or sport = :9030 )' || true
fi

echo "[3/6] Create or reuse the Docker network and named volumes"
docker network inspect "$DORIS_NETWORK" >/dev/null 2>&1 || docker network create "$DORIS_NETWORK"
docker volume inspect "$FE_VOLUME" >/dev/null 2>&1 || docker volume create "$FE_VOLUME"
docker volume inspect "$BE_VOLUME" >/dev/null 2>&1 || docker volume create "$BE_VOLUME"

echo "[4/6] Pull the pinned Doris image"
docker pull "$DORIS_IMAGE"

echo "[5/6] Start or reuse the Doris container"
if docker container inspect "$DORIS_CONTAINER" >/dev/null 2>&1; then
  CURRENT_IMAGE=$(docker inspect --format '{{.Config.Image}}' "$DORIS_CONTAINER")
  if [ "$CURRENT_IMAGE" != "$DORIS_IMAGE" ]; then
    echo "Container $DORIS_CONTAINER uses $CURRENT_IMAGE, expected $DORIS_IMAGE"
    echo "The existing container was left unchanged. Rename it or choose another course container name."
    exit 7
  fi
  CURRENT_NETWORKS=$(docker inspect --format '{{range $name, $_ := .NetworkSettings.Networks}}{{println $name}}{{end}}' "$DORIS_CONTAINER")
  CURRENT_MOUNTS=$(docker inspect --format '{{range .Mounts}}{{printf "%s|%s\n" .Name .Destination}}{{end}}' "$DORIS_CONTAINER")
  CURRENT_PORTS=$(docker port "$DORIS_CONTAINER" 2>/dev/null || true)
  if ! grep -Fxq "$DORIS_NETWORK" <<<"$CURRENT_NETWORKS" \
     || ! grep -Fxq "$FE_VOLUME|/opt/apache-doris/fe/doris-meta" <<<"$CURRENT_MOUNTS" \
     || ! grep -Fxq "$BE_VOLUME|/opt/apache-doris/be/storage" <<<"$CURRENT_MOUNTS" \
     || ! grep -Eq '9030/tcp.*127\.0\.0\.1:9030$' <<<"$CURRENT_PORTS" \
     || ! grep -Eq '8030/tcp.*127\.0\.0\.1:8030$' <<<"$CURRENT_PORTS" \
     || ! grep -Eq '8040/tcp.*127\.0\.0\.1:8040$' <<<"$CURRENT_PORTS"; then
    echo "Container $DORIS_CONTAINER does not use the expected network, named volumes, or localhost port bindings."
    echo "The existing container was left unchanged. Use a clean course container before continuing."
    exit 8
  fi
  if [ "$(docker inspect --format '{{.State.Status}}' "$DORIS_CONTAINER")" != "running" ]; then
    docker start "$DORIS_CONTAINER"
  fi
else
  docker run -d \
    --name "$DORIS_CONTAINER" \
    --network "$DORIS_NETWORK" \
    -p 127.0.0.1:9030:9030 \
    -p 127.0.0.1:8030:8030 \
    -p 127.0.0.1:8040:8040 \
    -v "$FE_VOLUME:/opt/apache-doris/fe/doris-meta" \
    -v "$BE_VOLUME:/opt/apache-doris/be/storage" \
    "$DORIS_IMAGE"
fi

echo "[6/6] Show the running course container"
docker ps --filter "name=^/${DORIS_CONTAINER}$"
""", title="Prepare the Doris environment");


**Connect to Doris and verify the nodes**

The Bash commands above start the container, but a running container is not necessarily ready to accept queries. The next cell waits until Docker reports `healthy`, opens a MySQL-compatible protocol connection to FE port 9030, and queries the FE and BE status tables. This confirms that both Doris services are ready before the Lab creates any database objects.

In [ ]:
lab.connect(container="doris", host="127.0.0.1", port=9030)

lab.sql("SELECT VERSION() AS doris_version, CURRENT_USER() AS current_user", title="Connection")
lab.sql("SHOW FRONTENDS", title="Frontend status")
lab.sql("SHOW BACKENDS", title="Backend status");


**Expected result**

- The Docker command finishes successfully and the `doris` container becomes `healthy`.
- `SHOW FRONTENDS` returns one row with `Alive=true`, `Join=true`, and `IsMaster=true`.
- `SHOW BACKENDS` returns one row with `Alive=true` and `SystemDecommissioned=false`.

**Common problems**

- If `server=` is empty or the active Docker socket does not exist on macOS, the cell opens Docker Desktop and waits for up to two minutes. Complete any first-run dialog, wait for the engine to become ready, and rerun the cell if the timeout expires.
- If Linux cannot connect to `/var/run/docker.sock`, start Docker Engine with `sudo systemctl start docker`. A `permission denied` message means your user needs the Docker permissions required by your organization.
- If a port is already in use and no matching course container owns it, identify its owner with `lsof -i :9030` on macOS or `ss -ltnp` on Linux. This Lab does not automatically reuse an unrelated local Doris installation because its version, topology, storage, and restart behavior may differ.
- If the image pull fails, check Docker Hub access and your proxy or VPN settings.
- If Doris becomes unhealthy, inspect `docker logs --tail 200 doris`.
- If `docker context show` or `DOCKER_HOST` points to an unavailable remote engine, select the intended Docker context or correct `DOCKER_HOST` before rerunning the cell.
- If Docker reports insufficient memory or disk space, increase Docker's resource allocation or managed disk capacity before pulling the image or importing the dataset.

## 3. Create the course database

A Doris Catalog contains databases, and each database contains tables. Create `doris_course` in the built-in `internal` Catalog, select it as the current database, and confirm that it is available.

In [ ]:
lab.execute("CREATE DATABASE IF NOT EXISTS doris_course")
lab.execute("USE doris_course")
lab.sql("SHOW DATABASES", title="Databases")
lab.sql("SELECT DATABASE() AS current_database", title="Current database");


**Expected result:** `SHOW DATABASES` includes `doris_course`, and `SELECT DATABASE()` returns `doris_course`.

## 4. Inspect the remote dataset in S3

Amazon S3 stores the Parquet files outside Doris. The S3 table-valued function (S3 TVF) lets a query read those files as rows and columns without first creating a Doris table.

The sample contains:

| Column | Meaning |
|---|---|
| `event_time` | Time at which the event occurred |
| `event_id` | Unique identifier for the event row |
| `user_id` | Identifier for the user |
| `event_type` | `view`, `cart`, or `purchase` |
| `region` | Synthetic region assigned consistently to the user |
| `product_id` | Identifier for the product |
| `revenue` | Purchase price, or `0.00` for non-purchase events |

In [ ]:
lab.preview_s3("""
SELECT
    event_time,
    event_id,
    user_id,
    event_type,
    region,
    product_id,
    revenue
FROM {{COURSE_S3}}
LIMIT 10
""");


**Expected result:** one table containing ten event rows. These rows are read directly from the course copy in S3; no rows have been written to the Doris internal table yet.

## 5. Create the first Doris internal table

An internal table stores its data in BE storage. Define the column names, data types, nullability, and a single replica for this one-BE sandbox. Then inspect the table definition returned by Doris.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS events (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL DEFAULT "0.00"
)
PROPERTIES (
    "replication_num" = "1"
)
""")

lab.sql(
    "DESC events",
    title="events column schema",
    columns=["Field", "Type", "Null", "Key", "Default"],
)

lab.sql(
    "SHOW CREATE TABLE events",
    title="Resolved table design",
    metadata_view="design",
);

**Expected result**

The column schema contains seven rows, one for each field created above. The design summary shows only the physical choices Doris resolved from this concise DDL:

| Design choice | Expected value |
|---|---|
| Table model | Duplicate Key model |
| Sort key | `event_time, event_id, user_id` |
| Partitioning | No explicit partitioning |
| Distribution | Random |
| Buckets per partition | Auto |
| Replica allocation | 1 replica per tablet |

If a previous Lab run left rows in this persistent table, the next section removes them before performing a fresh import.

## 6. Reload and validate the baseline dataset

`TRUNCATE TABLE` removes any rows left by an earlier run while preserving the `events` table definition. The following `INSERT INTO ... SELECT` then reads every matching Parquet file through the S3 TVF and writes the complete dataset into Doris again. Running both statements makes the cost of the actual load visible on every Lab run. For this 8 GB single-node sandbox, the helper temporarily limits pipeline tasks and concurrent S3 file scanners during the insert, then restores the session settings. This reduces peak BE memory without changing the rows written.

| Stage | What happens | Stored in Doris? |
|---|---|---|
| Remote Parquet files | S3 stores 40 files containing the source rows. | No |
| S3 TVF result | Doris reads the files as a temporary relational result. | No |
| `events` internal table | `INSERT INTO` writes 10,158,080 rows to BE storage. | Yes |

After the import, the validation query checks the row count and time range. Together, these values confirm that all expected rows and the complete event-time span are present.

In [ ]:
lab.execute("TRUNCATE TABLE events")

lab.insert("""
INSERT INTO events (
    event_time,
    event_id,
    user_id,
    event_type,
    region,
    product_id,
    revenue
)
SELECT
    event_time,
    event_id,
    user_id,
    event_type,
    region,
    product_id,
    revenue
FROM {{COURSE_S3}}
""", title="Baseline S3 insert", low_memory_s3=True)

lab.verify_baseline("""
SELECT
    COUNT(*) AS row_count,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time
FROM events
""");

**Expected result**

| Check | Expected value |
|---|---:|
| `row_count` | 10,158,080 |
| `min_event_time` | 2019-12-01 00:00:12 |
| `max_event_time` | 2020-03-11 04:53:08 |

The insert output reports **10,158,080 affected rows** and its elapsed time. The validation table then confirms the persisted result. Import time varies with network and Docker resources and is not part of the correctness check.

## 7. Run the first analytical query

This query reduces more than ten million detail rows to three event-funnel totals. It demonstrates a complete analytical query with `GROUP BY`, `COUNT`, and `SUM`: purchase rows contribute revenue, while `view` and `cart` rows contribute `0.00`.

In [ ]:
lab.verify_event_counts("""
SELECT
    event_type,
    COUNT(*) AS event_count,
    SUM(revenue) AS total_revenue
FROM events
GROUP BY event_type
ORDER BY event_count DESC
""");


**Expected result**

| Event type | Event count | Total revenue |
|---|---:|---:|
| `view` | 9,437,184 | 0.00 |
| `cart` | 589,824 | 0.00 |
| `purchase` | 131,072 | 39,984,455.64 |

### Stop the Doris sandbox

Run this optional cell when you want to release the container's CPU and memory. It stops the Doris processes but keeps the container, image, FE metadata volume, and BE storage volume. Skip it if you are continuing directly to Lab 2.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris
docker inspect --format 'container={{.State.Status}}' doris
""", title="Stop the Doris sandbox");


**Expected result:** Docker reports `container=exited`. The `doris-fe-meta` and `doris-be-storage` named volumes remain available.

### Restart the Doris sandbox

Run this cell when you want to continue with the same sandbox. It starts the existing container, waits for Doris to become healthy, reconnects to the FE, and verifies that the `events` table survived the stop. Do not run the stop cell immediately before this unless you actually want to test recovery.

This restart cell is idempotent: it can be run when Docker Desktop and the container are stopped, starting, or already running. On macOS it opens Docker Desktop when necessary; on Linux, start Docker Engine before running the cell.


In [ ]:
lab.start_container("doris")

lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")
lab.verify_recovery("SELECT COUNT(*) AS recovered_rows FROM events");


**Expected result:** the container returns to `healthy`, and `recovered_rows` is **10,158,080**. This confirms that the named FE and BE volumes preserved the Lab dataset.

## Lab complete

You connected to Doris through the MySQL-compatible protocol, inspected FE and BE, queried remote Parquet files with the S3 TVF, loaded 10,158,080 rows into an internal table, ran an analytical aggregation, and configured Docker named volumes to preserve the dataset.

Optional extension: [Connect Metabase and build a dashboard](lab1_optional_metabase_dashboard.ipynb)

Official references: [All-In-One image](https://doris.apache.org/community/developer-guide/all-in-one-image/) · [S3 TVF](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/table-valued-functions/s3/) · [CREATE TABLE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/table/CREATE-TABLE/)